# 04 - Evaluation on Testing Day (Multi-Config)
## Evaluate all models on Testing Day data

Input: `cleaned_test_*K.pkl`, `models/xgboost_top10_*K.json`, `training_results_all.pkl`
Output: per-class metrics, confusion matrices, comparison table

In [ ]:
import pandas as pd
import numpy as np
import pickle, os, gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

CONFIGS = [10000, 30000, 50000, 100000]
DATA_DIR = '../data/'
MODEL_DIR = '../models/'
all_eval = {}
print('OK')

In [ ]:
# Load training results for label mappings
with open(os.path.join(DATA_DIR, 'training_results_all.pkl'), 'rb') as f:
    train_results = pickle.load(f)

for config in CONFIGS:
    suffix = f'{config//1000}K'
    test_path = os.path.join(DATA_DIR, f'cleaned_test_{suffix}.pkl')
    model_path = os.path.join(MODEL_DIR, f'xgboost_top10_{suffix}.json')
    
    if not os.path.exists(test_path) or not os.path.exists(model_path):
        print(f'SKIP {suffix}: files not found')
        continue
    if suffix not in train_results:
        print(f'SKIP {suffix}: no training results')
        continue
    
    print(f'\n{"="*70}')
    print(f'  EVALUATION: {suffix} (Top-10 model)')
    print(f'{"="*70}')
    
    # Load test data
    with open(test_path, 'rb') as f:
        test_data = pickle.load(f)
    X_test = test_data['X']
    y_test_raw = test_data['y']
    all_classes = test_data['classes']
    
    # Get training label map
    tr = train_results[suffix]
    label_map = tr['label_map']
    train_classes = tr['classes']
    top_10 = tr['top_10']
    
    # Filter test: only classes in training
    valid_mask = np.isin(y_test_raw, list(label_map.keys()))
    X_test_valid = X_test[valid_mask].reset_index(drop=True)
    y_test_valid_raw = y_test_raw[valid_mask]
    y_test = np.array([label_map[y] for y in y_test_valid_raw])
    
    excluded = y_test_raw[~valid_mask]
    if len(excluded) > 0:
        exc_classes = [all_classes[i] for i in np.unique(excluded)]
        print(f'  Excluded (not in training): {exc_classes} ({len(excluded):,} samples)')
    
    print(f'  Test samples: {len(X_test_valid):,} | Classes: {len(train_classes)}')
    
    # Load Top-10 model
    model = XGBClassifier()
    model.load_model(model_path)
    
    # Select Top-10 features
    valid_features = [f for f in top_10 if f in X_test_valid.columns]
    X_test_top10 = X_test_valid[valid_features]
    
    # Predict
    y_pred = model.predict(X_test_top10)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f'  Accuracy:    {acc*100:.2f}%')
    print(f'  F1 (macro):  {f1_macro*100:.2f}%')
    print(f'  F1 (weighted): {f1_weighted*100:.2f}%')
    
    # Per-class report
    present = sorted(set(y_test) | set(y_pred))
    present_names = [train_classes[i] for i in present]
    print(f'\n{classification_report(y_test, y_pred, labels=present, target_names=present_names, digits=3)}')
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred, labels=present)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=present_names, yticklabels=present_names)
    plt.title(f'Confusion Matrix - {suffix} (Top-10)')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, f'confusion_matrix_{suffix}.png'), dpi=150)
    plt.show()
    
    all_eval[suffix] = {'accuracy': acc, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted,
                        'test_samples': len(y_test), 'classes': present_names}
    
    del X_test, X_test_valid, test_data; gc.collect()

with open(os.path.join(DATA_DIR, 'evaluation_results_all.pkl'), 'wb') as f:
    pickle.dump(all_eval, f)
print(f'\nSaved: evaluation_results_all.pkl')

In [ ]:
# Final comparison table
print(f'\n{"="*70}')
print('  FINAL COMPARISON: Testing Day Evaluation (Top-10 Model)')
print(f'{"="*70}')
print(f'{"Config":>8s} {"Samples":>10s} {"Accuracy%":>12s} {"F1 Macro%":>12s} {"F1 Weighted%":>14s}')
print('-'*60)
for k, v in all_eval.items():
    print(f'{k:>8s} {v["test_samples"]:>10,} {v["accuracy"]*100:>12.2f} {v["f1_macro"]*100:>12.2f} {v["f1_weighted"]*100:>14.2f}')
print(f'{"="*70}')
print('\n=== All notebooks complete! ===')